In [ ]:
#!/usr/bin/env python3
"""
ASL Alphabet Classification - Landmark-Based Experiments
=========================================================

This notebook is a companion to asl-experiments-kaggle.ipynb.
Instead of feeding raw pixels into a CNN, we:

  1. Run MediaPipe Hands on every training image to extract 21 hand
     landmarks (x, y, z per point = 63 features per image).
  2. Normalize those landmarks so they are invariant to hand position
     and scale in the frame (centre on wrist, divide by span).
  3. Train lightweight classifiers on the 63-feature vectors.

WHY LANDMARKS INSTEAD OF PIXELS?
---------------------------------
The CNN experiments achieve ~99.99 % on the Kaggle test set, but the
training images have clean white backgrounds.  On a live webcam the
background is cluttered, so the CNN collapses to a few high-confidence
classes (e.g. B, W).  Landmark coordinates carry no background
information, so a landmark classifier should generalise much better to
real-world conditions.

EXPERIMENTS
-----------
1) mlp_small  - 2-layer MLP  (128 -> 64 -> 29)
2) mlp_large  - 3-layer MLP  (256 -> 128 -> 64 -> 29)

Both use the same normalised landmark features and are evaluated with
the same metrics as the CNN notebook so results are directly comparable.

Images where MediaPipe cannot detect a hand are skipped during
feature extraction (typically < 2 % of the dataset).
"""

# Install mediapipe if running on Kaggle
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "mediapipe==0.10.5", "-q"])

In [ ]:
# ============================================================
# Section 1 - Imports
# ============================================================

import os
import csv
import json
import random
from pathlib import Path

import numpy as np
import cv2
import mediapipe as mp

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset

from sklearn.metrics import classification_report, confusion_matrix, f1_score

import pandas as pd
from tqdm import tqdm

print("PyTorch:", torch.__version__)
print("MediaPipe:", mp.__version__)

In [ ]:
# ============================================================
# Section 2 - Reproducibility and device
# ============================================================

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# ============================================================
# Section 3 - Output directory setup
# ============================================================

BASE_OUTPUT_DIR = Path("/kaggle/working/landmark_outputs")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = ["mlp_small", "mlp_large"]

print("Output dir:", BASE_OUTPUT_DIR)

In [ ]:
# ============================================================
# Section 4 - Landmark extraction
# ============================================================

# MediaPipe returns 21 landmarks, each with x, y, z -> 63 raw features
N_LANDMARKS = 21
N_RAW_FEATURES = N_LANDMARKS * 3  # 63

_mp_hands = mp.solutions.hands


def normalize_landmarks(landmarks_flat: np.ndarray) -> np.ndarray:
    """
    Make landmarks invariant to hand position and scale.

    Steps:
      1. Reshape to (21, 3).
      2. Subtract the wrist landmark (index 0) so the hand is
         centred at the origin.
      3. Divide by the distance between the wrist and the middle
         fingertip (landmark 12) so the scale is normalised.
      4. Flatten back to (63,).

    If the wrist-to-tip distance is zero (degenerate case), the raw
    centred values are returned unchanged.
    """
    pts = landmarks_flat.reshape(N_LANDMARKS, 3)
    pts = pts - pts[0]          # centre on wrist
    scale = np.linalg.norm(pts[12])  # distance wrist -> middle fingertip
    if scale > 1e-6:
        pts = pts / scale
    return pts.flatten()


def extract_landmarks_from_image(image_path: str, hands_detector) -> np.ndarray | None:
    """
    Run MediaPipe on one image and return the normalised 63-feature vector.
    Returns None if no hand is detected.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result = hands_detector.process(img_rgb)
    if not result.multi_hand_landmarks:
        return None
    lm = result.multi_hand_landmarks[0].landmark
    raw = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32).flatten()
    return normalize_landmarks(raw)


def extract_all_landmarks(data_dir: str):
    """
    Walk data_dir (one subfolder per class) and extract landmarks for
    every image.  Images where MediaPipe finds no hand are skipped.

    Returns
    -------
    features : np.ndarray, shape (N, 63)
    labels   : np.ndarray, shape (N,)  -- integer class indices
    class_names : list[str]
    skipped  : int  -- number of images with no hand detected
    """
    class_names = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    features, labels = [], []
    skipped = 0

    with _mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5,
    ) as detector:
        for class_name in tqdm(class_names, desc="Extracting landmarks"):
            class_dir = os.path.join(data_dir, class_name)
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                path = os.path.join(class_dir, fname)
                feat = extract_landmarks_from_image(path, detector)
                if feat is None:
                    skipped += 1
                    continue
                features.append(feat)
                labels.append(class_to_idx[class_name])

    return (
        np.array(features, dtype=np.float32),
        np.array(labels, dtype=np.int64),
        class_names,
        skipped,
    )

In [ ]:
# ============================================================
# Section 5 - Run extraction
# ============================================================

data_dir = "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train"

features, labels, class_names, skipped = extract_all_landmarks(data_dir)

num_classes = len(class_names)
print(f"Extracted: {len(features)} samples, {skipped} skipped (no hand detected)")
print(f"Feature shape: {features.shape}")
print(f"Classes ({num_classes}): {class_names}")

In [ ]:
# ============================================================
# Section 6 - Train / val / test split
# ============================================================

# Mirror the same 70/15/15 split used in asl-experiments-kaggle.ipynb
# using the same seed so the comparison is fair.

full_dataset = TensorDataset(
    torch.from_numpy(features),
    torch.from_numpy(labels),
)

n = len(full_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

In [ ]:
# ============================================================
# Section 7 - MLP model definitions
# ============================================================

class MLPSmall(nn.Module):
    """2-hidden-layer MLP: 63 -> 128 -> 64 -> num_classes"""
    def __init__(self, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class MLPLarge(nn.Module):
    """3-hidden-layer MLP: 63 -> 256 -> 128 -> 64 -> num_classes"""
    def __init__(self, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def build_model(model_name: str, num_classes: int) -> nn.Module:
    if model_name == "mlp_small":
        return MLPSmall(num_classes)
    if model_name == "mlp_large":
        return MLPLarge(num_classes)
    raise ValueError(f"Unknown model: {model_name}")

In [ ]:
# ============================================================
# Section 8 - Training and evaluation helpers
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        loss = criterion(out, y)
        total_loss += loss.item() * len(y)
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += len(y)
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(y.cpu().tolist())
    return total_loss / total, correct / total, all_preds, all_targets


def run_experiment(
    model_name: str,
    train_ds, val_ds, test_ds,
    class_names: list,
    num_classes: int,
    device: str,
    epochs: int = 30,
    batch_size: int = 256,
    lr: float = 1e-3,
):
    print(f"\n{'='*60}")
    print(f"  Experiment: {model_name}")
    print(f"{'='*60}")

    out_dir = BASE_OUTPUT_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size)

    model = build_model(model_name, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss,   "val_acc": val_acc,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} "
                  f"| val loss {val_loss:.4f} acc {val_acc:.4f}")

    # Reload best checkpoint
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / "best_model.pt")

    # Final test evaluation
    test_loss, test_acc, preds, targets = evaluate(model, test_loader, criterion, device)
    macro_f1    = f1_score(targets, preds, average="macro",    zero_division=0)
    weighted_f1 = f1_score(targets, preds, average="weighted", zero_division=0)

    print(f"\n  Test loss: {test_loss:.6f}  acc: {test_acc:.4f}  macro-F1: {macro_f1:.4f}")

    # Save artefacts
    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)

    report = classification_report(targets, preds, target_names=class_names, zero_division=0)
    (out_dir / "classification_report.txt").write_text(report)

    np.save(out_dir / "confusion_matrix.npy", confusion_matrix(targets, preds))

    metrics = {
        "model":         model_name,
        "test_loss":     test_loss,
        "test_accuracy": test_acc,
        "macro_f1":      macro_f1,
        "weighted_f1":   weighted_f1,
        "num_classes":   num_classes,
    }
    return metrics, history

In [ ]:
# ============================================================
# Section 9 - Run all experiments
# ============================================================

all_metrics = []

for model_name in EXPERIMENTS:
    set_seed(SEED)
    metrics, history = run_experiment(
        model_name=model_name,
        train_ds=train_ds,
        val_ds=val_ds,
        test_ds=test_ds,
        class_names=class_names,
        num_classes=num_classes,
        device=device,
        epochs=30,
    )
    all_metrics.append(metrics)

summary_df = pd.DataFrame(all_metrics).sort_values("test_accuracy", ascending=False).reset_index(drop=True)

summary_df.to_csv(BASE_OUTPUT_DIR / "all_experiments_summary.csv", index=False)
summary_df.to_json(BASE_OUTPUT_DIR / "all_experiments_summary.json", orient="records", indent=2)

print("\nFinal summary:")
display(summary_df)

In [ ]:
# ============================================================
# Section 10 - Report-friendly table
# ============================================================

report_table = summary_df.copy()
for col in ["test_loss", "test_accuracy", "macro_f1", "weighted_f1"]:
    report_table[col] = report_table[col].map(lambda x: round(float(x), 4))

display(report_table)